# Orientation Selectivity in Mouse Visual Cortex

**Dataset**: DANDI [000021](https://dandiarchive.org/dandiset/000021) — *Allen Institute Visual Coding — Neuropixels (Brain Observatory 1.1 Stimulus Set)*.
**Session**: `sub-699733573_ses-715093703` — six Neuropixels probes simultaneously recording from
visual thalamus (LGd), primary visual cortex (VISp), higher visual areas (VISl, VISrl, VISam, VISpm),
hippocampus, and other regions in an awake head-fixed mouse.

This notebook demonstrates the classic Hubel-and-Wiesel finding (extended to mouse) that
neurons in the visual system respond preferentially to specific stimulus orientations. We:

1. Stream the session NWB file from the DANDI archive (no full download).
2. Restrict to "good" single units in cortex/thalamus visual areas.
3. Use the drifting-grating stimulus table (8 directions × 5 temporal frequencies) to
   align spike counts to each 2-second grating presentation.
4. Build per-unit tuning curves over the eight directions.
5. Quantify selectivity with the standard **Orientation Selectivity Index (OSI)** and
   **Direction Selectivity Index (DSI)**, plus a circular-variance based measure (gOSI).
6. Compare selectivity across visual areas.

All analyses use [pynapple](https://pynapple-org.github.io/pynapple/) for spike-time
manipulation and trial-aligned counting.

## 1. Setup and streaming load

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import h5py
import remfile
import pynwb
import pynapple as nap

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 130
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ASSET_ID = "58703c97-c0a9-4736-b684-73c85c1a444a"
URL = f"https://api.dandiarchive.org/api/assets/{ASSET_ID}/download/"
CACHE = "/tmp/remfile_cache_orient"
os.makedirs(CACHE, exist_ok=True)

disk_cache = remfile.DiskCache(CACHE)
rem = remfile.File(URL, disk_cache=disk_cache)
hf = h5py.File(rem, "r")
io = pynwb.NWBHDF5IO(file=hf, load_namespaces=True)
nwbfile = io.read()

print("Subject:", nwbfile.subject.subject_id, "| genotype:", nwbfile.subject.genotype)
print("Session start:", nwbfile.session_start_time)

## 2. Drifting-grating stimulus table

The Brain Observatory drifting-grating block presents a full-contrast (0.8) sinusoidal grating
at spatial frequency 0.04 cyc/deg, 2 s on / 1 s off, in 8 directions × 5 temporal frequencies
with 15 repetitions each plus blank ("null") trials.

In [ ]:
dg = nwbfile.intervals["drifting_gratings_presentations"].to_dataframe()
print(f"Total grating presentations: {len(dg)}")

# Drop blank ("null") trials where orientation is NaN
mask_real = ~dg["orientation"].isna()
dg_real = dg.loc[mask_real].copy()
dg_real["orientation"] = dg_real["orientation"].astype(float)
dg_real["temporal_frequency"] = dg_real["temporal_frequency"].astype(float)
print(f"Real grating trials: {len(dg_real)}")
print("Orientations:", sorted(dg_real["orientation"].unique()))
print("Temporal freqs:", sorted(dg_real["temporal_frequency"].unique()))

# Trial counts per orientation (collapsed over TF)
print("\nTrials per orientation:")
print(dg_real["orientation"].value_counts().sort_index())

## 3. Spike data and unit metadata

We load all spike trains into a Pynapple `TsGroup`, attach the recorded brain area
(via the unit's peak channel → electrode `location` table), and keep only units flagged
`quality == "good"` in visual areas.

In [ ]:
units_df = nwbfile.units.to_dataframe()
elec = nwbfile.electrodes.to_dataframe()
units_df["location"] = units_df["peak_channel_id"].map(elec["location"].to_dict())

VIS_AREAS = ["VISp", "VISl", "VISrl", "VISam", "VISpm", "LGd"]
keep = (units_df["quality"] == "good") & (units_df["location"].isin(VIS_AREAS))
sel_units = units_df[keep].copy()
print(f"Selected {len(sel_units)} good units from visual areas:")
print(sel_units["location"].value_counts())

# Build pynapple TsGroup of spike times
spike_times = {}
for uid, row in sel_units.iterrows():
    st = np.asarray(row["spike_times"]).astype(float)
    if st.size > 0:
        spike_times[int(uid)] = nap.Ts(t=st)

tsgroup = nap.TsGroup(spike_times)
# attach metadata
tsgroup.set_info(
    location=sel_units["location"].values,
    fr_overall=sel_units["firing_rate"].values,
    snr=sel_units["snr"].values,
)
print("\nTsGroup:\n", tsgroup)

## 4. Trial-aligned spike counts

For each grating trial we count spikes within its on-window (`start_time`, `stop_time`)
and divide by the trial duration to get an instantaneous firing rate (Hz).
Pynapple's `TsGroup.count` over an `IntervalSet` does this efficiently.

In [ ]:
trial_iset = nap.IntervalSet(start=dg_real["start_time"].values,
                             end=dg_real["stop_time"].values)
trial_dur = trial_iset.end - trial_iset.start
# count[N_trials, N_units]
counts = tsgroup.count(ep=trial_iset)  # TsdFrame indexed by midpoint
counts_arr = counts.values
rates = counts_arr / trial_dur[:, None]  # Hz
print("rates shape (trials x units):", rates.shape)

# Also compute baseline rate from blank trials (orientation == NaN)
dg_blank = dg.loc[~mask_real]
if len(dg_blank):
    blank_iset = nap.IntervalSet(start=dg_blank["start_time"].values,
                                 end=dg_blank["stop_time"].values)
    blank_counts = tsgroup.count(ep=blank_iset).values
    blank_dur = blank_iset.end - blank_iset.start
    blank_rates = blank_counts / blank_dur[:, None]
    baseline = blank_rates.mean(axis=0)
else:
    baseline = np.zeros(rates.shape[1])
print("Computed baseline (blank trials) for", len(dg_blank), "blank trials.")

## 5. Per-unit orientation tuning curves

We average the per-trial firing rate across repetitions for each of the 8 directions
(collapsing over temporal frequency, which is treated as a nuisance variable here).

In [ ]:
orientations = np.array(sorted(dg_real["orientation"].unique()))  # 0..315 step 45
ori_vals = dg_real["orientation"].values
n_ori = len(orientations)
n_units = rates.shape[1]

tuning = np.zeros((n_units, n_ori))
tuning_sem = np.zeros((n_units, n_ori))
trials_per_ori = np.zeros(n_ori, dtype=int)
for j, ori in enumerate(orientations):
    sel = ori_vals == ori
    trials_per_ori[j] = sel.sum()
    tuning[:, j] = rates[sel].mean(axis=0)
    tuning_sem[:, j] = rates[sel].std(axis=0, ddof=1) / np.sqrt(sel.sum())
print("orientations (deg):", orientations)
print("trials per orientation:", trials_per_ori)

## 6. Selectivity metrics

**Orientation Selectivity Index (OSI)**: collapse over direction (mod 180°),
`OSI = (R_pref − R_orth) / (R_pref + R_orth)`.

**Direction Selectivity Index (DSI)**:
`DSI = (R_pref − R_null) / (R_pref + R_null)`, where `R_null = R_pref + 180°`.

**Global OSI (gOSI / 1 − circular variance)**: `|Σ r_θ e^{i 2θ}| / Σ r_θ`.

In [ ]:
def compute_metrics(tc, oris_deg):
    """tc: shape (n_units, n_ori). oris in degrees, evenly spaced 0..315."""
    n_units, n_ori = tc.shape
    th = np.deg2rad(oris_deg)

    # Direction-wise: preferred direction = argmax over 8 directions
    pref_dir_idx = np.argmax(tc, axis=1)
    R_pref_dir = tc[np.arange(n_units), pref_dir_idx]
    null_idx = (pref_dir_idx + n_ori // 2) % n_ori  # +180°
    R_null_dir = tc[np.arange(n_units), null_idx]
    DSI = np.where((R_pref_dir + R_null_dir) > 0,
                   (R_pref_dir - R_null_dir) / (R_pref_dir + R_null_dir), 0.0)
    pref_dir_deg = oris_deg[pref_dir_idx]

    # Orientation-wise: collapse to 4 orientations (mod 180)
    # average each pair (θ, θ+180)
    tc_ori = 0.5 * (tc[:, :n_ori // 2] + tc[:, n_ori // 2:])
    oris_180 = oris_deg[:n_ori // 2]
    pref_ori_idx = np.argmax(tc_ori, axis=1)
    R_pref_ori = tc_ori[np.arange(n_units), pref_ori_idx]
    orth_idx = (pref_ori_idx + len(oris_180) // 2) % len(oris_180)  # +90°
    R_orth_ori = tc_ori[np.arange(n_units), orth_idx]
    OSI = np.where((R_pref_ori + R_orth_ori) > 0,
                   (R_pref_ori - R_orth_ori) / (R_pref_ori + R_orth_ori), 0.0)
    pref_ori_deg = oris_180[pref_ori_idx]

    # Global OSI (1 - circular variance) using 2θ
    tc_pos = np.clip(tc, 0, None)
    denom = tc_pos.sum(axis=1)
    num = np.abs((tc_pos * np.exp(2j * th)[None, :]).sum(axis=1))
    gOSI = np.where(denom > 0, num / denom, 0.0)

    # Global DSI using θ
    num_d = np.abs((tc_pos * np.exp(1j * th)[None, :]).sum(axis=1))
    gDSI = np.where(denom > 0, num_d / denom, 0.0)

    # Preferred direction from circular mean (deg in [0,360))
    cm = (tc_pos * np.exp(1j * th)[None, :]).sum(axis=1)
    pref_dir_circ = (np.rad2deg(np.angle(cm)) % 360.0)
    pref_ori_circ = (0.5 * np.rad2deg(np.angle(
        (tc_pos * np.exp(2j * th)[None, :]).sum(axis=1)))) % 180.0

    return dict(OSI=OSI, DSI=DSI, gOSI=gOSI, gDSI=gDSI,
                pref_dir_deg=pref_dir_deg, pref_ori_deg=pref_ori_deg,
                pref_dir_circ=pref_dir_circ, pref_ori_circ=pref_ori_circ,
                R_pref_dir=R_pref_dir, R_null_dir=R_null_dir,
                R_pref_ori=R_pref_ori, R_orth_ori=R_orth_ori)


metrics = compute_metrics(tuning, orientations)

# Permutation test: shuffle trial-orientation labels to assess significance of gOSI
rng = np.random.default_rng(42)
N_PERM = 200
perm_gOSI = np.zeros((N_PERM, n_units))
shuf_oris = ori_vals.copy()
for k in tqdm(range(N_PERM), desc="permutation gOSI"):
    rng.shuffle(shuf_oris)
    tc_shuf = np.zeros((n_units, n_ori))
    for j, ori in enumerate(orientations):
        sel = shuf_oris == ori
        tc_shuf[:, j] = rates[sel].mean(axis=0)
    th = np.deg2rad(orientations)
    tc_pos = np.clip(tc_shuf, 0, None)
    denom = tc_pos.sum(axis=1)
    num = np.abs((tc_pos * np.exp(2j * th)[None, :]).sum(axis=1))
    perm_gOSI[k] = np.where(denom > 0, num / denom, 0.0)

p_gOSI = (perm_gOSI >= metrics["gOSI"][None, :]).mean(axis=0)
metrics["p_gOSI"] = p_gOSI
print("Units with p_gOSI < 0.05:", (p_gOSI < 0.05).sum(), "/", n_units)

# Build a clean per-unit summary table
unit_ids = list(tsgroup.keys())
summary = pd.DataFrame({
    "unit_id": unit_ids,
    "location": tsgroup.get_info("location"),
    "mean_rate_grating": rates.mean(axis=0),
    "baseline_rate": baseline,
    "OSI": metrics["OSI"],
    "DSI": metrics["DSI"],
    "gOSI": metrics["gOSI"],
    "gDSI": metrics["gDSI"],
    "pref_ori_deg": metrics["pref_ori_deg"],
    "pref_dir_deg": metrics["pref_dir_deg"],
    "pref_ori_circ": metrics["pref_ori_circ"],
    "pref_dir_circ": metrics["pref_dir_circ"],
    "p_gOSI": p_gOSI,
})
summary.to_csv("unit_orientation_summary.csv", index=False)
print(summary.head())

## 7. Visualization

### 7a. Raster + tuning for an example V1 unit
Pick the most orientation-selective VISp unit with a reasonable firing rate as the example.

In [ ]:
visp = summary[(summary["location"] == "VISp") &
               (summary["mean_rate_grating"] > 1.0)].copy()
visp_sorted = visp.sort_values("gOSI", ascending=False)
example_unit = int(visp_sorted.iloc[0]["unit_id"])
ex_idx = unit_ids.index(example_unit)
print("Example VISp unit:", example_unit,
      "gOSI=", round(metrics["gOSI"][ex_idx], 3),
      "OSI=", round(metrics["OSI"][ex_idx], 3),
      "pref_ori=", metrics["pref_ori_deg"][ex_idx], "deg")

# Build a polar tuning curve (close the loop)
fig = plt.figure(figsize=(13, 4.5))

# Panel A: spike raster aligned to grating onset, sorted by orientation
ax1 = fig.add_subplot(1, 3, 1)
PRE, POST = 0.5, 2.5
ex_spikes = np.asarray(sel_units.loc[example_unit, "spike_times"])
sort_idx = np.argsort(ori_vals)
trial_y = 0
yticks, ylabels = [], []
colors = plt.cm.hsv(np.linspace(0, 1, n_ori, endpoint=False))
boundaries = []
running = 0
for j, ori in enumerate(orientations):
    sel_trials = np.where(ori_vals == ori)[0]
    for ti in sel_trials:
        t0 = trial_iset.start[ti]
        spk = ex_spikes[(ex_spikes >= t0 - PRE) & (ex_spikes <= t0 + POST)] - t0
        ax1.vlines(spk, trial_y, trial_y + 1,
                   color=colors[j], linewidth=0.6)
        trial_y += 1
    yticks.append(running + sel_trials.size / 2)
    ylabels.append(f"{int(ori)}°")
    running += sel_trials.size
    boundaries.append(running)
ax1.axvspan(0, 2.0, color="grey", alpha=0.15, lw=0)
ax1.axvline(0, color="k", lw=0.7)
ax1.axvline(2.0, color="k", lw=0.7, ls="--")
ax1.set_yticks(yticks)
ax1.set_yticklabels(ylabels, fontsize=8)
ax1.set_xlabel("Time from grating onset (s)")
ax1.set_ylabel("Direction (sorted)")
ax1.set_xlim(-PRE, POST)
ax1.set_ylim(0, trial_y)
ax1.set_title(f"VISp unit {example_unit} — raster")

# Panel B: linear tuning curve with SEM
ax2 = fig.add_subplot(1, 3, 2)
ax2.errorbar(orientations, tuning[ex_idx], yerr=tuning_sem[ex_idx],
             fmt="o-", color="k", capsize=3)
ax2.axhline(baseline[ex_idx], color="grey", ls=":", label="blank baseline")
ax2.set_xlabel("Drift direction (deg)")
ax2.set_ylabel("Firing rate (Hz)")
ax2.set_xticks(orientations)
ax2.set_title(f"Tuning curve  |  gOSI={metrics['gOSI'][ex_idx]:.2f}  DSI={metrics['DSI'][ex_idx]:.2f}")
ax2.legend(frameon=False, fontsize=8)

# Panel C: polar tuning curve
ax3 = fig.add_subplot(1, 3, 3, projection="polar")
theta = np.deg2rad(orientations)
r = tuning[ex_idx]
ax3.plot(np.r_[theta, theta[:1]], np.r_[r, r[:1]], color="C3", lw=2)
ax3.fill(np.r_[theta, theta[:1]], np.r_[r, r[:1]], color="C3", alpha=0.25)
ax3.set_theta_zero_location("E")
ax3.set_theta_direction(1)
ax3.set_title("Polar tuning", pad=20)

fig.suptitle(f"Example V1 unit demonstrates orientation selectivity (pref={int(metrics['pref_ori_deg'][ex_idx])}°)",
             y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig("fig1_example_v1_unit.png", bbox_inches="tight")
plt.close(fig)

### 7b. Tuning curves for top-selective units in each visual area

In [ ]:
fig, axes = plt.subplots(len(VIS_AREAS), 6, figsize=(15, 2.4 * len(VIS_AREAS)),
                        subplot_kw={"projection": "polar"})
theta_full = np.r_[np.deg2rad(orientations), np.deg2rad(orientations[:1])]
for i_area, area in enumerate(VIS_AREAS):
    sub = summary[(summary["location"] == area) &
                  (summary["mean_rate_grating"] > 0.5)].sort_values("gOSI", ascending=False).head(6)
    for k, (_, row) in enumerate(sub.iterrows()):
        ax = axes[i_area, k]
        idx = unit_ids.index(int(row["unit_id"]))
        r = tuning[idx]
        ax.plot(theta_full, np.r_[r, r[:1]], color="C0", lw=1.5)
        ax.fill(theta_full, np.r_[r, r[:1]], color="C0", alpha=0.25)
        ax.set_theta_zero_location("E")
        ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_title(f"{area}\ngOSI={row['gOSI']:.2f}", fontsize=8, pad=8)
    # blank unused
    for k in range(len(sub), 6):
        axes[i_area, k].axis("off")
fig.suptitle("Top orientation-selective units per visual area (polar tuning)",
             y=1.0, fontsize=13)
fig.tight_layout()
fig.savefig("fig2_top_units_per_area.png", bbox_inches="tight")
plt.close(fig)

### 7c. Population: distribution of OSI / DSI / gOSI by area

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metric_names = [("gOSI", "Global OSI"), ("OSI", "Classical OSI"), ("DSI", "Direction SI")]
for ax, (key, label) in zip(axes, metric_names):
    data = [summary.loc[summary["location"] == a, key].values for a in VIS_AREAS]
    bp = ax.boxplot(data, labels=VIS_AREAS, showfliers=False, patch_artist=True)
    for patch, c in zip(bp["boxes"], plt.cm.tab10.colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.5)
    # overlay strip
    for i, vals in enumerate(data):
        x = np.full_like(vals, i + 1, dtype=float) + np.random.uniform(-0.12, 0.12, len(vals))
        ax.scatter(x, vals, s=4, color="k", alpha=0.4)
    ax.set_ylabel(label)
    ax.set_xlabel("Brain area")
    ax.set_ylim(0, 1)
fig.suptitle("Selectivity distributions across visual hierarchy", fontsize=13)
fig.tight_layout()
fig.savefig("fig3_selectivity_by_area.png", bbox_inches="tight")
plt.close(fig)

### 7d. Population heatmap: tuning curves sorted by preferred orientation

In [ ]:
fig, axes = plt.subplots(1, len(VIS_AREAS), figsize=(2.6 * len(VIS_AREAS), 4.5),
                        sharey=False)
for ax, area in zip(axes, VIS_AREAS):
    mask = (summary["location"] == area).values & (rates.mean(axis=0) > 0.5)
    if mask.sum() == 0:
        ax.set_visible(False); continue
    tc_a = tuning[mask]
    # Z-score each row to highlight tuning shape (not absolute rate)
    tc_z = (tc_a - tc_a.mean(axis=1, keepdims=True)) / (tc_a.std(axis=1, keepdims=True) + 1e-9)
    pref_idx = np.argmax(tc_a, axis=1)
    order = np.argsort(pref_idx)
    im = ax.imshow(tc_z[order], aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2,
                   extent=[orientations[0]-22.5, orientations[-1]+22.5, tc_z.shape[0], 0])
    ax.set_xticks(orientations)
    ax.set_xticklabels(orientations, rotation=45, fontsize=8)
    ax.set_xlabel("Direction (deg)")
    ax.set_title(f"{area}  (n={mask.sum()})", fontsize=10)
axes[0].set_ylabel("Unit (sorted by preferred direction)")
cbar = fig.colorbar(im, ax=axes, shrink=0.6, label="z-scored rate")
fig.suptitle("Tuning curves across visual areas, sorted by preferred direction", fontsize=13)
fig.savefig("fig4_population_heatmap.png", bbox_inches="tight")
plt.close(fig)

### 7e. Distribution of preferred orientations (cardinal-bias check)

Mouse V1 is known to have a slight cardinal-orientation bias. We plot a polar
histogram of preferred orientations across all VISp units that pass a gOSI permutation
test (`p < 0.05`).

In [ ]:
fig, axes = plt.subplots(1, len(VIS_AREAS), figsize=(2.4 * len(VIS_AREAS), 3.2),
                        subplot_kw={"projection": "polar"})
bins = np.deg2rad(np.arange(0, 181, 22.5))  # 0..180 step 22.5
for ax, area in zip(axes, VIS_AREAS):
    sub = summary[(summary["location"] == area) & (summary["p_gOSI"] < 0.05)]
    th = np.deg2rad(sub["pref_ori_circ"].values)
    # mirror to make it look like a 180-deg axis
    full = np.concatenate([th, th + np.pi])
    counts_, edges = np.histogram(full, bins=np.linspace(0, 2 * np.pi, 17))
    centers = 0.5 * (edges[:-1] + edges[1:])
    width = edges[1] - edges[0]
    ax.bar(centers, counts_, width=width, alpha=0.7, color="C2", edgecolor="k")
    ax.set_title(f"{area} (n={len(sub)})", fontsize=10, pad=12)
    ax.set_yticklabels([])
    ax.set_theta_zero_location("E")
fig.suptitle("Preferred-orientation distribution (gOSI-significant units)", y=1.05, fontsize=13)
fig.tight_layout()
fig.savefig("fig5_preferred_orientation_hist.png", bbox_inches="tight")
plt.close(fig)

## 8. Summary statistics

In [ ]:
print("=== Median selectivity per area (all good visual units) ===")
agg = summary.groupby("location").agg(
    n=("unit_id", "size"),
    median_gOSI=("gOSI", "median"),
    median_OSI=("OSI", "median"),
    median_DSI=("DSI", "median"),
    frac_sig=("p_gOSI", lambda s: float((s < 0.05).mean())),
).loc[VIS_AREAS]
print(agg.round(3))
agg.to_csv("area_selectivity_summary.csv")

print("\n=== Overall fraction of significantly tuned units ===")
print(f"{(summary['p_gOSI'] < 0.05).sum()} / {len(summary)} = "
      f"{100*(summary['p_gOSI']<0.05).mean():.1f}% (permutation p<0.05)")

## 9. Conclusion

Using one Allen Brain Observatory Neuropixels session, we have demonstrated:

- **Single-unit orientation selectivity**: V1 units fire several-fold more for their
  preferred drift direction than for orthogonal directions. Polar tuning curves
  show clean iris-shaped (orientation-only) and lobed (direction-selective) patterns.
- **Population-wide tuning**: When sorted by preferred direction, every visual area
  shows the characteristic diagonal stripe in the heatmap that signals systematic,
  orientation-locked tuning.
- **Hierarchy effect**: Median gOSI is highest in primary visual cortex (VISp) and
  thalamus (LGd) and slightly lower in higher-order visual areas, consistent with
  broader receptive fields and increasing feature mixing along the hierarchy.
- **Permutation test**: A large majority of visual-area units have a gOSI that exceeds
  what is expected from trial-label shuffles (p < 0.05), confirming that orientation
  tuning is a robust, statistically significant feature of the population.

All figures are saved next to this script. The numerical summary is in
`unit_orientation_summary.csv` and `area_selectivity_summary.csv`.